# Notebook 28: Phase Collapse Under Perturbation
Testing robustness of zeta–GUE phase under perturbations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(0)

## Generate synthetic proxy data

In [ ]:
def make_data(n=500):
    zeta = rng.normal(0, 1, (n,2))
    gue = zeta + rng.normal(0, 0.3, (n,2))
    poisson = rng.normal(2, 1.5, (n,2))
    return zeta, gue, poisson

zeta, gue, poisson = make_data()

## Perturbations

In [ ]:
def global_shuffle(x):
    return x[np.random.permutation(len(x))]

def add_noise(x, eps):
    return x * (1 + eps * np.random.normal(size=x.shape))

zeta_shuffle = global_shuffle(zeta)
zeta_noise = add_noise(zeta, 0.1)

## Simple classifier overlap proxy

In [ ]:
def overlap_score(a, b):
    X = np.vstack([a,b])
    y = np.array([0]*len(a) + [1]*len(b))
    Xtr, Xte, ytr, yte = train_test_split(X,y,test_size=0.3,random_state=0)
    clf = KNeighborsClassifier(n_neighbors=5)
    clf.fit(Xtr,ytr)
    return clf.score(Xte,yte)

pairs = {
    "zeta vs GUE": overlap_score(zeta, gue),
    "zeta vs Poisson": overlap_score(zeta, poisson),
    "zeta(shuffle) vs GUE": overlap_score(zeta_shuffle, gue),
    "zeta(noise) vs GUE": overlap_score(zeta_noise, gue)
}

pairs

## Plot comparison

In [ ]:
labels = list(pairs.keys())
values = list(pairs.values())

plt.figure()
plt.bar(labels, values)
plt.xticks(rotation=30)
plt.ylabel("accuracy (proxy overlap)")
plt.title("Phase collapse under perturbation")
plt.show()